# Corrigé · Exercice 4.1 : quel modèle pour HelpDesk-Ops ?

> Notebook autonome. ~40 minutes de calcul sur le poste CPU de préparation.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
import time

import pandas as pd

from helpdesk_ops import evaluation as ev
from helpdesk_ops.config import basculer_vllm, creer_llm

pd.set_option("display.width", 180)
dataset = langfuse.get_dataset(ev.synchroniser_dataset(langfuse))

# Les trois candidats. gemma4 : raisonnement coupé (208 s contre 16 s par décision, mesuré en préparation)
CANDIDATS = {
    "granite4.1:3b": creer_llm("small"),
    "granite4.1:8b": creer_llm("agent"),
    "gemma4:12b": creer_llm("alt", reasoning=False),
}
# vLLM (salle) : sur 24 Go, gemma4 ne tient pas à côté des deux granite. Avant chaque modèle :
# basculer_vllm(MODE_VLLM[modele]) (1 à 3 min pour passer d'un mode à l'autre ; sans effet sous Ollama)
MODE_VLLM = {"granite4.1:3b": "agents", "granite4.1:8b": "agents", "gemma4:12b": "juge"}
# Session en 4 jours, si le temps manque : décommenter pour sauter gemma4 (et la bascule en mode juge)
# del CANDIDATS["gemma4:12b"]


def experience(nom: str, modele: str, tache, evaluateurs: list):
    debut = time.perf_counter()
    resultat = dataset.run_experiment(name=nom, run_name=f"{nom} · {modele}", task=tache, evaluators=evaluateurs,
                                      run_evaluators=[ev.ev_moyennes], max_concurrency=1, metadata={"modele": modele})
    print(f"   {nom:<17} {modele:<15} {time.perf_counter() - debut:5.0f} s")
    return resultat

## 1. Les experiments

In [ ]:
resultats = {}
for modele, candidat in CANDIDATS.items():
    basculer_vllm(MODE_VLLM[modele])
    resultats[("routage", modele)] = experience("helpdesk-routage", modele, ev.tache_routage(candidat), [ev.ev_routage, ev.ev_latence])
    resultats[("outils", modele)] = experience("helpdesk-outils", modele, ev.tache_outil(candidat),
                                               [ev.ev_outil_valide, ev.ev_outil_attendu, ev.ev_latence])
basculer_vllm("agents")
langfuse.flush()

## 2. Le tableau comparatif

In [ ]:
lignes = []
for modele in CANDIDATS:
    r, o = ev.resume(resultats[("routage", modele)]), ev.resume(resultats[("outils", modele)])
    lignes.append({"modèle": modele,
                   "routage juste": r.get("routage_juste"), "latence routage p95 (s)": r.get("latence_p95_s"),
                   "outil valide": o.get("outil_valide"), "outil attendu": o.get("outil_attendu"),
                   "latence outil p95 (s)": o.get("latence_p95_s")})
tableau = pd.DataFrame(lignes)
print(tableau.to_string(index=False))

## 3. Les cas ratés du meilleur modèle

In [ ]:
meilleur = tableau.sort_values(["routage juste", "outil attendu"], ascending=False).iloc[0]["modèle"]
print("Meilleur sur ces deux scores :", meilleur)
l = pd.DataFrame(ev.lignes_resultat(resultats[("routage", meilleur)]))
print("\nRoutage raté :")
print(l[l["routage_juste"] == 0][["cas", "specialiste", "refus", "origine"]].fillna("").to_string(index=False))
l = pd.DataFrame(ev.lignes_resultat(resultats[("outils", meilleur)]))
print("\nOutil inattendu :")
print(l[l.get("outil_attendu", pd.Series(1, index=l.index)) == 0][["cas", "outils", "origine"]].fillna("").to_string(index=False))

## 4. Recommandation (à adapter aux chiffres affichés ci-dessus)

La recommandation se construit **à partir du tableau**, pas avant. Grille de lecture :

1. **Écarter** un modèle dont `outil_valide` est nettement plus bas : des appels mal formés cassent l'agent, quel que soit le reste.
2. **Comparer** `routage juste` et `outil attendu` en gardant l'incertitude en tête : sur 20 cas, un écart de 1 cas = 5 points, souvent non significatif.
3. **Pondérer par la latence sur le matériel cible** : les p95 mesurés ici sont sur CPU ; en salle (GPU), refaire au moins un run par modèle avant de conclure.
4. **Vérifier les cas graves** : le poème (C19) est un échec du garde-fou par règles, identique pour tous les modèles ; il ne départage pas les modèles, mais il bloque la mise en production (red-team, cet après-midi).
5. **Prochaine mesure** : agrandir le dataset (50 à 100 cas, dont des cas réels anonymisés), ajouter la fidélité (juge J4-04) et la qualification.

**Mesuré pendant la préparation** (poste CPU, 21/10/2026, consigne de routage v2, un modèle chargé à la fois) :

| modèle | routage juste | outil valide | outil attendu | p95 outil (s) | durée des deux experiments |
|---|---|---|---|---|---|
| granite4.1:3b | 75 % | 93,8 % | 81,2 % | 31 | 47 s + 275 s |
| granite4.1:8b | **85 %** | **100 %** | **93,8 %** | 63 | 154 s + 378 s |
| gemma4:12b | 80 % | 87,5 % | 81,2 % | 111 | 650 s + 1 407 s |

Ce qui en ressort : `granite4.1:8b` est en tête sur les trois scores de qualité, mais les écarts de routage (75 / 80 / 85 %) tiennent en **deux cas** sur 20 ; le plus gros modèle n'est pas le meilleur ici, et il est le plus lent. Les échecs de routage du 8b (C19, C08, C03) sont ceux du garde-fou et d'une annotation discutable, pas du modèle.

Exemple de formulation : « *Nous recommandons X : routage juste à N %, outil attendu à M % (contre … pour Y), p95 de … s sur CPU. Limite : 20 cas, écarts de l'ordre de l'incertitude ; à confirmer sur 100 cas et sur GPU. Bloquant avant production : le contournement du garde-fou (C19).* »